# GeoAgent + ChatGPT/Codex OAuth

This notebook uses the same ChatGPT/Codex OAuth flow as the QGIS plugin, but stores the login in GeoAgent's user config directory for regular Python and Jupyter sessions.

- Install: `pip install "GeoAgent[openai]"`.
- Run the login cell once. Later notebooks can reuse the stored login automatically.

In [ ]:
# %pip install -q "GeoAgent[openai]"

In [ ]:
from geoagent import ensure_openai_codex_environment
from geoagent import login_openai_codex

try:
    ensure_openai_codex_environment()
    print("Using stored ChatGPT/Codex login.")
except RuntimeError:
    login_openai_codex()
    print("Logged in with ChatGPT/Codex OAuth.")

In [ ]:
import os

MODEL = os.environ.get("OPENAI_CODEX_MODEL", "gpt-5.5")
print("Using model:", MODEL)

In [ ]:
from geoagent import GeoAgent, GeoAgentConfig

agent = GeoAgent(
    config=GeoAgentConfig(
        provider="openai-codex",
        model=MODEL,
        temperature=0.0,
        max_tokens=2048,
    ),
)

resp = agent.chat("In one sentence, what is GeoAgent useful for?")
print("success:", resp.success)
print(resp.answer_text)
print("executed_tools:", resp.executed_tools)

## Image Example

In [ ]:
# %pip install -q pillow

In [ ]:
from pathlib import Path

from IPython.display import Image, display
from PIL import Image as PILImage
from PIL import ImageDraw

image_path = Path("codex_example_map.png")
img = PILImage.new("RGB", (640, 360), "#dce9f2")
draw = ImageDraw.Draw(img)
draw.rectangle((0, 235, 640, 360), fill="#6fbf73")
draw.polygon([(0, 90), (190, 70), (310, 160), (0, 230)], fill="#8ccf6a")
draw.polygon([(430, 45), (640, 25), (640, 230), (520, 205)], fill="#7db56f")
draw.line(
    [(70, 0), (120, 80), (170, 140), (230, 210), (330, 360)],
    fill="#3d7fbf",
    width=18,
)
draw.ellipse((100, 95, 140, 135), fill="#e84d4d")
draw.ellipse((455, 155, 495, 195), fill="#f2c94c")
draw.text((32, 28), "Synthetic map image", fill="#1f2937")
draw.text((145, 105), "site A", fill="#1f2937")
draw.text((500, 165), "site B", fill="#1f2937")
img.save(image_path)
display(Image(filename=str(image_path)))

In [ ]:
image_bytes = image_path.read_bytes()
resp = agent.chat(
    [
        {
            "text": "Describe this map-like image. Identify the labeled sites, water, and land cover."
        },
        {
            "image": {
                "format": "png",
                "source": {"bytes": image_bytes},
            }
        },
    ]
)
print(resp.answer_text)

## Leafmap Example

In [ ]:
# %pip install -q "GeoAgent[leafmap]" leafmap

In [ ]:
import leafmap.maplibregl as leafmap
from geoagent import for_leafmap

m = leafmap.Map(center=[-83.92, 35.96], zoom=9, height="520px")
m

In [ ]:
map_agent = for_leafmap(
    m,
    provider="openai-codex",
    model_id=MODEL,
    fast=True,
)

resp = map_agent.chat("Using the current map state, describe the map center and zoom.")
print(resp.answer_text)
print("tools:", resp.executed_tools)